# Hedonic Pricing Regression — RQ1 & RQ2

**RQ1:** What property, host, and location characteristics are the strongest predictors of nightly Airbnb listing price in London?

**RQ2:** Does London borough moderate the relationship between listing characteristics and nightly price?

This notebook runs the same logic as `hedonic_pricing_model.py` (RQ1) and `moderation_analysis.py` (RQ2) — it imports those scripts directly rather than duplicating the code, so the notebook and the committed pipeline scripts can never silently drift apart. Run this notebook from the same folder as those two files and the dataset CSV.

## Setup

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 140)

import hedonic_pricing_model as hpm
import moderation_analysis as mod

print(f"Reusing config/functions from hedonic_pricing_model.py and moderation_analysis.py")
print(f"INPUT_FILE = {hpm.INPUT_FILE}")

Reusing config/functions from hedonic_pricing_model.py and moderation_analysis.py
INPUT_FILE = airbnb_london_full33_pricebands_20260804_1347_features.csv


## Load and prepare data

In [2]:
df = hpm.load_dataset(hpm.INPUT_FILE)
df = hpm.prepare_model_data(df)
print(f"Rows: {len(df)}")
df.head()

Rows: 2519


,title,price_raw,price_gbp_total,price_gbp_per_night,rating,review_count,is_new_listing,url,room_id,property_type,...,search_borough,zone,price_band_min,price_band_max,price_band_label,page_scraped,scraped_at,listing_type,log_review_count,log_price
0,Room in Barking,£145 £125 total Show price breakdown,125.0,31.25,5.00,6.0,False,https://www.airbnb.co.uk/rooms/1656014065513845102?check_in=2026-09-01&check_out=2026-09-05&search_mode=regular_sear...,1656014065513845102,Room,...,"Barking and Dagenham, London, United Kingdom",Outer London,NaN,75.0,£0-75,1,2026-08-04T13:47:46,Room,1.945910,3.442019
1,Room in Newham,£202 £178 total Show price breakdown Pay £0 today Free cancellation,178.0,44.50,4.84,64.0,False,https://www.airbnb.co.uk/rooms/18605858?check_in=2026-09-01&check_out=2026-09-05&search_mode=regular_search&category...,18605858,Room,...,"Barking and Dagenham, London, United Kingdom",Outer London,NaN,75.0,£0-75,1,2026-08-04T13:47:46,Room,4.174387,3.795489
2,Room in East Ham,£184 total Show price breakdown £184 total Pay £0 today Free cancellation,184.0,46.00,4.68,282.0,False,https://www.airbnb.co.uk/rooms/35714817?check_in=2026-09-01&check_out=2026-09-05&search_mode=regular_search&category...,35714817,Room,...,"Barking and Dagenham, London, United Kingdom",Outer London,NaN,75.0,£0-75,1,2026-08-04T13:47:46,Room,5.645447,3.828641
3,Room in Newham,£234 total Show price breakdown £234 total,234.0,58.50,NaN,NaN,True,https://www.airbnb.co.uk/rooms/830259732159297926?check_in=2026-09-01&check_out=2026-09-05&search_mode=regular_searc...,830259732159297926,Room,...,"Barking and Dagenham, London, United Kingdom",Outer London,NaN,75.0,£0-75,1,2026-08-04T13:47:46,Room,0.000000,4.069027
4,Room in Barking,£164 £141 total Show price breakdown Pay £0 today Free cancellation,141.0,35.25,4.33,80.0,False,https://www.airbnb.co.uk/rooms/1393315971091496785?check_in=2026-09-01&check_out=2026-09-05&search_mode=regular_sear...,1393315971091496785,Room,...,"Barking and Dagenham, London, United Kingdom",Outer London,NaN,75.0,£0-75,1,2026-08-04T13:47:46,Room,4.394449,3.562466


## RQ1 — Main hedonic pricing model

The sample is split into four groups (peer-to-peer vs hotel-brand, rated vs new/unrated) rather than pooled into one model. An earlier pooled version zero-filled `rating` for unrated listings and used an `is_new_listing` dummy — VIF diagnostics showed that produced near-perfect structural collinearity (VIF > 120) between the two, since one was almost entirely predictable from the other by construction. Splitting the sample removes that artifact entirely.

In [3]:
is_hotel = df["is_hotel_brand"].astype(bool)
is_new = df["is_new_listing"].astype(bool)

rated_formula = hpm.build_formula(dv="log_price", categorical=hpm.CATEGORICAL_PREDICTORS,
                                    numeric=hpm.RATED_NUMERIC_PREDICTORS, dummy=[])
unrated_formula = hpm.build_formula(dv="log_price", categorical=hpm.CATEGORICAL_PREDICTORS,
                                      numeric=hpm.UNRATED_NUMERIC_PREDICTORS, dummy=[])

groups = {
    "MAIN MODEL (peer-to-peer, rated)": (~is_hotel & ~is_new, rated_formula),
    "SUPPLEMENTARY (peer-to-peer, new/unrated)": (~is_hotel & is_new, unrated_formula),
    "SUPPLEMENTARY (hotel-brand, rated)": (is_hotel & ~is_new, rated_formula),
    "NOT MODELLED (hotel-brand, new/unrated)": (is_hotel & is_new, None),
}

pd.DataFrame({"group": list(groups.keys()), "n": [m.sum() for m, _ in groups.values()]})

,group,n
0,"MAIN MODEL (peer-to-peer, rated)",2096
1,"SUPPLEMENTARY (peer-to-peer, new/unrated)",308
2,"SUPPLEMENTARY (hotel-brand, rated)",103
3,"NOT MODELLED (hotel-brand, new/unrated)",12


### RQ1 main model (peer-to-peer, rated listings) — answers RQ1

In [4]:
main_mask, main_formula = groups["MAIN MODEL (peer-to-peer, rated)"]
main_df = df.loc[main_mask].copy()
main_model = smf.ols(formula=main_formula, data=main_df).fit()
print(main_model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.617
Model:                            OLS   Adj. R-squared:                  0.615
Method:                 Least Squares   F-statistic:                     305.1
Date:                Tue, 11 Aug 2026   Prob (F-statistic):               0.00
Time:                        09:21:43   Log-Likelihood:                -1540.3
No. Observations:                2096   AIC:                             3105.
Df Residuals:                    2084   BIC:                             3172.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

**VIF diagnostics — proposal's success criterion is VIF < 5:**

In [5]:
y, X = dmatrices(main_formula, data=main_df, return_type="dataframe")
vif_main = pd.DataFrame({
    "predictor": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif_main = vif_main.loc[vif_main["predictor"] != "Intercept"].sort_values("VIF", ascending=False)
vif_main

,predictor,VIF
2,C(listing_type)[T.Flat],2.696267
8,C(listing_type)[T.Room],2.568545
5,C(listing_type)[T.Home],1.843229
11,log_review_count,1.196577
6,C(listing_type)[T.Other],1.171023
7,C(listing_type)[T.Place to stay],1.123233
3,C(listing_type)[T.Guest house],1.108344
4,C(listing_type)[T.Guest suite],1.072546
9,C(listing_type)[T.Townhouse],1.065524
1,C(zone)[T.Outer London],1.047117


In [6]:
print(f"Adjusted R-squared: {main_model.rsquared_adj:.4f}")
print(f"Max VIF: {vif_main['VIF'].max():.2f}  (threshold: 5)")
print("All predictors below VIF 5 -> no multicollinearity concern, LASSO robustness check not required."
      if vif_main['VIF'].max() < 5 else "VIF exceeds threshold -> run LASSO as robustness check.")

Adjusted R-squared: 0.6149
Max VIF: 2.70  (threshold: 5)
All predictors below VIF 5 -> no multicollinearity concern, LASSO robustness check not required.


### Supplementary models (peer-to-peer new/unrated; hotel-brand rated)

In [7]:
supplementary_results = {}
for label, (mask, formula) in groups.items():
    if label == "MAIN MODEL (peer-to-peer, rated)" or formula is None:
        continue
    group_df = df.loc[mask].copy()
    if len(group_df) < hpm.MIN_ROWS_TO_MODEL:
        print(f"Skipping {label}: only {len(group_df)} rows.")
        continue
    m = smf.ols(formula=formula, data=group_df).fit()
    supplementary_results[label] = m
    print(f"\n=== {label} (N={len(group_df)}) ===")
    print(m.summary())


=== SUPPLEMENTARY (peer-to-peer, new/unrated) (N=308) ===
                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.542
Model:                            OLS   Adj. R-squared:                  0.529
Method:                 Least Squares   F-statistic:                     39.25
Date:                Tue, 11 Aug 2026   Prob (F-statistic):           1.12e-45
Time:                        09:21:43   Log-Likelihood:                -208.97
No. Observations:                 308   AIC:                             437.9
Df Residuals:                     298   BIC:                             475.2
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------

## RQ2 — Does borough moderate the listing-type price relationship?

Four specifications, fit on the same RQ1 main-model sample (peer-to-peer, rated listings):

1. **Baseline** — zone, no interaction (same as the RQ1 main model)
2. **Borough main effects** — full 33-borough control, no interaction
3. **Full moderation (literal RQ2 spec)** — `search_borough × listing_type`
4. **Well-identified fallback** — `zone × listing_type`

The proposal's own risk register (Section 4.3) anticipates that the full 33-borough interaction may be poorly identified given how many dummy interactions it produces relative to sample size, and pre-commits to falling back to Inner/Outer zone if so.

In [8]:
main_df2 = df.loc[~is_hotel & ~is_new].copy()

m1_formula, m2_formula, m3_formula, m4_formula = mod.build_moderation_formulas()

model1 = smf.ols(formula=m1_formula, data=main_df2).fit()
model2 = smf.ols(formula=m2_formula, data=main_df2).fit()
model3 = smf.ols(formula=m3_formula, data=main_df2).fit()
model4 = smf.ols(formula=m4_formula, data=main_df2).fit()

comparison = pd.DataFrame({
    "model": ["1: zone (baseline)", "2: borough (no interaction)",
              "3: borough x listing_type (literal RQ2)", "4: zone x listing_type (fallback)"],
    "params": [int(m.df_model) + 1 for m in (model1, model2, model3, model4)],
    "adj_r2": [m.rsquared_adj for m in (model1, model2, model3, model4)],
    "aic": [m.aic for m in (model1, model2, model3, model4)],
})
comparison

,model,params,adj_r2,aic
0,1: zone (baseline),12,0.614915,3104.627377
1,2: borough (no interaction),43,0.647420,2950.376045
2,3: borough x listing_type (literal RQ2),219,0.686510,2868.216621
3,4: zone x listing_type (fallback),20,0.620452,3082.210285


### Checking whether Model 3 (full borough interaction) is actually trustworthy

Model 3 wins on AIC/adjusted R² — but that alone doesn't mean the interaction is real. Each interaction term is checked against how many listings actually back that specific borough × listing-type cell, since OLS can fit a deceptively "significant" line through just 1–2 points.

In [9]:
ct = pd.crosstab(main_df2["search_borough"], main_df2["listing_type"])
sparse_cells = int((ct <= 1).sum().sum())
print(f"borough x listing_type grid: {ct.shape[0]} boroughs x {ct.shape[1]} listing types = {ct.size} cells")
print(f"{sparse_cells} cells ({100*sparse_cells/ct.size:.0f}%) have 0 or 1 listings")

borough x listing_type grid: 33 boroughs x 9 listing types = 297 cells
138 cells (46%) have 0 or 1 listings


In [10]:
def interaction_reliability(model, ct, min_cell_n=5):
    interaction_terms = model.params.index[model.params.index.str.contains(":")]
    def cell_n(term):
        borough_part, listing_part = term.split(":")
        b = borough_part.split("[T.")[1].rstrip("]")
        l = listing_part.split("[T.")[1].rstrip("]")
        return ct.loc[b, l] if (b in ct.index and l in ct.columns) else 0
    rows = []
    for t in interaction_terms:
        n = cell_n(t)
        rows.append({"term": t, "cell_n": n, "coef": model.params[t],
                     "se": model.bse[t], "p": model.pvalues[t],
                     "reliable": n >= min_cell_n and model.pvalues[t] < 0.05})
    return pd.DataFrame(rows)

rel3 = interaction_reliability(model3, ct)
print(f"Model 3: {len(rel3)} interaction terms, {(rel3['cell_n'] < 5).sum()} backed by fewer than 5 listings.")
rel3.loc[rel3['reliable']].sort_values('p')

Model 3: 256 interaction terms, 172 backed by fewer than 5 listings.


,term,cell_n,coef,se,p,reliable
199,"C(search_borough)[T.Ealing, London, United Kingdom]:C(listing_type)[T.Room]",24,0.959378,0.218184,0.000012,True
206,"C(search_borough)[T.Havering, London, United Kingdom]:C(listing_type)[T.Room]",12,1.273014,0.298857,0.000021,True
208,"C(search_borough)[T.Hounslow, London, United Kingdom]:C(listing_type)[T.Room]",20,1.224263,0.287538,0.000022,True
221,"C(search_borough)[T.Waltham Forest, London, United Kingdom]:C(listing_type)[T.Room]",19,0.962386,0.227489,0.000024,True
103,"C(search_borough)[T.Ealing, London, United Kingdom]:C(listing_type)[T.Home]",17,1.012466,0.248387,0.000048,True
204,"C(search_borough)[T.Haringey, London, United Kingdom]:C(listing_type)[T.Room]",17,0.875424,0.241143,0.000291,True
100,"C(search_borough)[T.Camden, London, United Kingdom]:C(listing_type)[T.Home]",6,1.163633,0.326858,0.000380,True
202,"C(search_borough)[T.Hackney, London, United Kingdom]:C(listing_type)[T.Room]",32,0.768706,0.222721,0.000570,True
196,"C(search_borough)[T.Camden, London, United Kingdom]:C(listing_type)[T.Room]",30,0.853390,0.261548,0.001123,True
205,"C(search_borough)[T.Harrow, London, United Kingdom]:C(listing_type)[T.Room]",18,0.683022,0.210986,0.001228,True


### Model 4 (zone × listing_type) — the well-identified RQ2 result

In [11]:
zone_ct = pd.crosstab(main_df2["zone"], main_df2["listing_type"])
rel4 = interaction_reliability(model4, zone_ct)
print(model4.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.624
Model:                            OLS   Adj. R-squared:                  0.620
Method:                 Least Squares   F-statistic:                     181.2
Date:                Tue, 11 Aug 2026   Prob (F-statistic):               0.00
Time:                        09:21:43   Log-Likelihood:                -1521.1
No. Observations:                2096   AIC:                             3082.
Df Residuals:                    2076   BIC:                             3195.
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                                               coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------

In [12]:
print(f"Model 1 (baseline) adj. R-squared: {model1.rsquared_adj:.4f}, AIC: {model1.aic:.1f}")
print(f"Model 4 (zone x listing_type) adj. R-squared: {model4.rsquared_adj:.4f}, AIC: {model4.aic:.1f}")
print()
rel4

Model 1 (baseline) adj. R-squared: 0.6149, AIC: 3104.6
Model 4 (zone x listing_type) adj. R-squared: 0.6205, AIC: 3082.2



,term,cell_n,coef,se,p,reliable
0,C(zone)[T.Outer London]:C(listing_type)[T.Flat],459,0.007979,0.075263,0.915583,False
1,C(zone)[T.Outer London]:C(listing_type)[T.Guest house],26,0.553342,0.374940,0.140146,False
2,C(zone)[T.Outer London]:C(listing_type)[T.Guest suite],16,-0.191377,0.382619,0.617003,False
3,C(zone)[T.Outer London]:C(listing_type)[T.Home],202,-0.019875,0.099675,0.841974,False
4,C(zone)[T.Outer London]:C(listing_type)[T.Other],24,0.754861,0.162712,0.000004,True
5,C(zone)[T.Outer London]:C(listing_type)[T.Place to stay],19,0.607558,0.196819,0.002049,True
6,C(zone)[T.Outer London]:C(listing_type)[T.Room],381,0.075796,0.077868,0.330475,False
7,C(zone)[T.Outer London]:C(listing_type)[T.Townhouse],10,0.340113,0.247426,0.169402,False


## Conclusion

**RQ1:** The main hedonic model (peer-to-peer, rated listings, N=2,096) achieves an adjusted R-squared well above the proposal's own 0.35 success threshold, with all VIF values below 5 — no multicollinearity concern, so no LASSO robustness check is triggered. `listing_type` and `zone` are the strongest predictors of price.

**RQ2:** The literal 33-borough × listing_type interaction (Model 3) technically wins on fit statistics, but 67% of its interaction terms are backed by fewer than 5 listings — an overfitting artifact, not genuine moderation, exactly as the proposal's own risk register anticipated. The well-identified fallback (Model 4, zone × listing_type) is reported as the primary RQ2 finding: a modest but genuine improvement over the no-interaction baseline, with the moderation effect concentrated in minor listing-type categories rather than the dominant ones (Flat/Room/Home).